In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Export Student Overview

In [ ]:

# ==========================================
# 1. LOAD DATA
# ==========================================

mahasiswa = pd.read_csv("../data/dim_mahasiswa.csv")
prodi = pd.read_csv("../data/dim_prodi.csv")
mata_kuliah = pd.read_csv("../data/dim_mata_kuliah.csv")


# ==========================================
# 2. SELECT REQUIRED COLUMNS
# ==========================================

mahasiswa_selected = mahasiswa[
    [
        "mahasiswa_key",
        "angkatan",
        "jenis_kelamin",
        "jalur_masuk_nama",
        "prodi_key",
        "is_active"
    ]
].copy()

prodi_selected = prodi[
    [
        "prodi_key",
        "nama_resmi",
        "jenjang_nama",
        "fakultas_nama",
        "jurusan_nama",
        "is_eksakta"
    ]
].copy()


# ==========================================
# 3. COUNT COURSES PER STUDY PROGRAM
# ==========================================

mata_kuliah_prodi = (
    mata_kuliah
    .groupby("prodi_key")
    .agg(
        total_mata_kuliah=("mata_kuliah_key", "nunique")
    )
    .reset_index()
)


# ==========================================
# 4. MERGE DATA
# ==========================================

student_overview = mahasiswa_selected.merge(
    prodi_selected,
    on="prodi_key",
    how="left",
    validate="many_to_one"
)

student_overview = student_overview.merge(
    mata_kuliah_prodi,
    on="prodi_key",
    how="left",
    validate="many_to_one"
)

student_overview["total_mata_kuliah"] = (
    student_overview["total_mata_kuliah"].fillna(0).astype(int)
)


# ==========================================
# 5. VALIDATION
# ==========================================

print("========== STUDENT OVERVIEW ==========")
print("Total records:", len(student_overview))
print("Unique students:", student_overview["mahasiswa_key"].nunique())
print("Missing faculty:", student_overview["fakultas_nama"].isna().sum())
print("Missing study program:", student_overview["nama_resmi"].isna().sum())
print("Duplicate student:", student_overview["mahasiswa_key"].duplicated().sum())

display(student_overview.head())


# ==========================================
# 6. EXPORT
# ==========================================

student_overview.to_csv(
    "../hasil/student_overview.csv",
    index=False
)

print("student_overview.csv berhasil dibuat!")


========== STUDENT OVERVIEW ==========
Total records: 300071
Unique students: 300071
Missing faculty: 0
Missing study program: 0
Duplicate student: 0


,mahasiswa_key,angkatan,jenis_kelamin,jalur_masuk_nama,prodi_key,is_active,nama_resmi,jenjang_nama,fakultas_nama,jurusan_nama,is_eksakta,total_mata_kuliah
0,214546,2023,L,SNBP,1353,1,TEKNOLOGI HASIL PERIKANAN - S1,Sarjana,ILMU KELAUTAN DAN PERIKANAN,NaN,1,118
1,214817,2023,P,SNBT,1353,1,TEKNOLOGI HASIL PERIKANAN - S1,Sarjana,ILMU KELAUTAN DAN PERIKANAN,NaN,1,118
2,214977,2023,L,SNBT,1353,1,TEKNOLOGI HASIL PERIKANAN - S1,Sarjana,ILMU KELAUTAN DAN PERIKANAN,NaN,1,118
3,215875,2023,P,SNBT,1353,1,TEKNOLOGI HASIL PERIKANAN - S1,Sarjana,ILMU KELAUTAN DAN PERIKANAN,NaN,1,118
4,216003,2023,L,Seleksi Mandiri UTBK,1353,1,TEKNOLOGI HASIL PERIKANAN - S1,Sarjana,ILMU KELAUTAN DAN PERIKANAN,NaN,1,118


student_overview.csv berhasil dibuat!


# Export Academic Performance

In [8]:

import pandas as pd

# ==========================================
# 1. LOAD DATA
# ==========================================

krs = pd.read_csv("../data/fact_krs.csv")
mahasiswa = pd.read_csv("../data/dim_mahasiswa.csv")
prodi = pd.read_csv("../data/dim_prodi.csv")
kelas = pd.read_csv("../data/dim_kelas_kuliah.csv")
mata_kuliah = pd.read_csv("../data/dim_mata_kuliah.csv")
semester = pd.read_csv("../data/dim_semester.csv")


# ==========================================
# 2. FILTER APPROVED KRS
# ==========================================

krs["is_approved"] = (
    krs["is_approved"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes", "y"])
)

krs_approved = krs[
    krs["is_approved"]
].copy()


# ==========================================
# 3. SELECT REQUIRED COLUMNS
# ==========================================

krs_selected = krs_approved[
    [
        "mahasiswa_key",
        "kelas_kuliah_key",
        "semester_key",
        "nilai_akhir",
        "is_approved"
    ]
].copy()

mahasiswa_selected = mahasiswa[
    [
        "mahasiswa_key",
        "angkatan",
        "jenis_kelamin",
        "prodi_key"
    ]
].copy()

prodi_selected = prodi[
    [
        "prodi_key",
        "nama_resmi",
        "fakultas_nama",
        "jenjang_nama"
    ]
].copy()

kelas_selected = kelas[
    [
        "kelas_kuliah_key",
        "mata_kuliah_key",
        "kapasitas",
        "hari"
    ]
].copy()

mata_kuliah_selected = mata_kuliah[
    [
        "mata_kuliah_key",
        "nama_resmi"
    ]
].copy()

semester_selected = semester[
    [
        "semester_key",
        "tahun",
        "kode"
    ]
].copy()


# ==========================================
# 4. MERGE DATA
# ==========================================

academic_performance = krs_selected.merge(
    mahasiswa_selected,
    on="mahasiswa_key",
    how="left",
    validate="many_to_one"
)

academic_performance = academic_performance.merge(
    prodi_selected,
    on="prodi_key",
    how="left",
    validate="many_to_one"
)

academic_performance = academic_performance.merge(
    kelas_selected,
    on="kelas_kuliah_key",
    how="left",
    validate="many_to_one"
)

academic_performance = academic_performance.merge(
    mata_kuliah_selected,
    on="mata_kuliah_key",
    how="left",
    validate="many_to_one",
    suffixes=("_prodi", "_mata_kuliah")
)

academic_performance = academic_performance.merge(
    semester_selected,
    on="semester_key",
    how="left",
    validate="many_to_one"
)


# ==========================================
# 5. CREATE ANALYSIS COLUMNS
# ==========================================

academic_performance["nilai_akhir"] = pd.to_numeric(
    academic_performance["nilai_akhir"],
    errors="coerce"
)

# Kategori nilai numerik
academic_performance["kategori_nilai"] = pd.cut(
    academic_performance["nilai_akhir"],
    bins=[-float("inf"), 60, 70, 80, 90, float("inf")],
    labels=[
        "Below 60",
        "60-69",
        "70-79",
        "80-89",
        "90-100"
    ],
    right=False
)

# Penanda nilai >= 60
academic_performance["lulus_60"] = (
    academic_performance["nilai_akhir"] >= 60
)


# ==========================================
# 6. VALIDATION
# ==========================================

print("========== ACADEMIC PERFORMANCE ==========")
print("Total approved KRS:", len(academic_performance))
print("Unique students:", academic_performance["mahasiswa_key"].nunique())
print("Missing scores:", academic_performance["nilai_akhir"].isna().sum())
print("Missing study program:", academic_performance["nama_resmi_prodi"].isna().sum())
print("Missing semester:", academic_performance["tahun"].isna().sum())

display(academic_performance.head())


# ==========================================
# 7. EXPORT
# ==========================================

academic_performance.to_csv(
    "../hasil/academic_performance.csv",
    index=False
)

print("academic_performance.csv berhasil dibuat!")


========== ACADEMIC PERFORMANCE ==========
Total approved KRS: 2849812
Unique students: 54052
Missing scores: 0
Missing study program: 0
Missing semester: 0


,mahasiswa_key,kelas_kuliah_key,semester_key,nilai_akhir,is_approved,angkatan,jenis_kelamin,prodi_key,nama_resmi_prodi,fakultas_nama,jenjang_nama,mata_kuliah_key,kapasitas,hari,nama_resmi_mata_kuliah,tahun,kode,kategori_nilai,lulus_60
0,20826,520249,54,79.82,True,2006,P,721,SISTEM-SISTEM PERTANIAN - S2,PROGRAM PASCASARJANA,Magister,25560,15,3,TEKNIK PENANGANAN BAHAN PANGAN,2016,20161,70-79,True
1,20826,520517,54,73.36,True,2006,P,721,SISTEM-SISTEM PERTANIAN - S2,PROGRAM PASCASARJANA,Magister,54901,3,3,Farming Systems,2016,20161,70-79,True
2,21372,520354,54,76.86,True,2006,L,721,SISTEM-SISTEM PERTANIAN - S2,PROGRAM PASCASARJANA,Magister,50055,19,2,Sistem Pertanian Berkelanjutan,2016,20161,70-79,True
3,21372,520472,54,70.39,True,2006,L,721,SISTEM-SISTEM PERTANIAN - S2,PROGRAM PASCASARJANA,Magister,52441,5,5,Pertanian Presisi,2016,20161,70-79,True
4,21372,520516,54,71.97,True,2006,L,721,SISTEM-SISTEM PERTANIAN - S2,PROGRAM PASCASARJANA,Magister,54900,21,2,Pertanian Presisi,2016,20161,70-79,True


academic_performance.csv berhasil dibuat!


# Export Attendance Analytics

In [9]:

import pandas as pd

# ==========================================
# 1. LOAD DATA
# ==========================================

presensi = pd.read_csv("../data/fact_presensi.csv")
tipe_kehadiran = pd.read_csv("../data/dim_tipe_kehadiran.csv")
mahasiswa = pd.read_csv("../data/dim_mahasiswa.csv")
prodi = pd.read_csv("../data/dim_prodi.csv")


# ==========================================
# 2. SELECT REQUIRED COLUMNS
# ==========================================

presensi_selected = presensi[
    [
        "mahasiswa_key",
        "kelas_kuliah_key",
        "tipe_kehadiran_key",
        "tanggal_pertemuan",
        "hadir_flag"
    ]
].copy()

tipe_selected = tipe_kehadiran[
    [
        "tipe_kehadiran_key",
        "nama"
    ]
].copy()

mahasiswa_selected = mahasiswa[
    [
        "mahasiswa_key",
        "angkatan",
        "jenis_kelamin",
        "prodi_key",
        "is_active"
    ]
].copy()

prodi_selected = prodi[
    [
        "prodi_key",
        "nama_resmi",
        "fakultas_nama",
        "jenjang_nama"
    ]
].copy()


# ==========================================
# 3. MERGE DATA
# ==========================================

attendance_analytics = presensi_selected.merge(
    tipe_selected,
    on="tipe_kehadiran_key",
    how="left",
    validate="many_to_one"
)

attendance_analytics = attendance_analytics.rename(
    columns={"nama": "tipe_kehadiran_nama"}
)

attendance_analytics = attendance_analytics.merge(
    mahasiswa_selected,
    on="mahasiswa_key",
    how="left",
    validate="many_to_one"
)

attendance_analytics = attendance_analytics.merge(
    prodi_selected,
    on="prodi_key",
    how="left",
    validate="many_to_one"
)


# ==========================================
# 4. DATA TYPE PREPARATION
# ==========================================

attendance_analytics["tanggal_pertemuan"] = pd.to_datetime(
    attendance_analytics["tanggal_pertemuan"],
    errors="coerce"
)

attendance_analytics["hadir_flag"] = (
    attendance_analytics["hadir_flag"]
    .astype(str)
    .str.lower()
    .map({
        "true": True,
        "1": True,
        "yes": True,
        "y": True,
        "false": False,
        "0": False,
        "no": False,
        "n": False
    })
)

# Tambahkan tahun dan bulan untuk analisis tren
attendance_analytics["tahun_presensi"] = (
    attendance_analytics["tanggal_pertemuan"].dt.year
)

attendance_analytics["bulan_presensi"] = (
    attendance_analytics["tanggal_pertemuan"].dt.month
)


# ==========================================
# 5. VALIDATION
# ==========================================

print("========== ATTENDANCE ANALYTICS ==========")

print("Total attendance records:",
      len(attendance_analytics))

print("Unique students:",
      attendance_analytics["mahasiswa_key"].nunique())

print("Missing attendance category:",
      attendance_analytics["tipe_kehadiran_nama"].isna().sum())

print("Missing study program:",
      attendance_analytics["nama_resmi"].isna().sum())

print("Missing attendance flag:",
      attendance_analytics["hadir_flag"].isna().sum())

print("\nAttendance category distribution:")
print(
    attendance_analytics["tipe_kehadiran_nama"]
    .value_counts(dropna=False)
)

display(attendance_analytics.head())


# ==========================================
# 6. EXPORT
# ==========================================

attendance_analytics.to_csv(
    "../hasil/attendance_analytics.csv",
    index=False
)

print("attendance_analytics.csv berhasil dibuat!")


========== ATTENDANCE ANALYTICS ==========
Total attendance records: 3000000
Unique students: 274882
Missing attendance category: 0
Missing study program: 0
Missing attendance flag: 0

Attendance category distribution:
tipe_kehadiran_nama
Hadir    2609774
Sakit     150372
Izin      149822
Absen      90032
Name: count, dtype: int64


,mahasiswa_key,kelas_kuliah_key,tipe_kehadiran_key,tanggal_pertemuan,hadir_flag,tipe_kehadiran_nama,angkatan,jenis_kelamin,prodi_key,is_active,nama_resmi,fakultas_nama,jenjang_nama,tahun_presensi,bulan_presensi
0,24995,733997,1,2026-11-20,True,Hadir,2007,P,93,0,ILMU PEMERINTAHAN - S1,ILMU SOSIAL DAN ILMU POLITIK,Sarjana,2026,11
1,49035,733997,1,2026-11-20,True,Hadir,2012,P,93,0,ILMU PEMERINTAHAN - S1,ILMU SOSIAL DAN ILMU POLITIK,Sarjana,2026,11
2,149405,733997,1,2026-11-20,True,Hadir,2020,L,93,0,ILMU PEMERINTAHAN - S1,ILMU SOSIAL DAN ILMU POLITIK,Sarjana,2026,11
3,214591,733997,1,2026-11-20,True,Hadir,2023,L,93,0,ILMU PEMERINTAHAN - S1,ILMU SOSIAL DAN ILMU POLITIK,Sarjana,2026,11
4,7358,734000,2,2026-11-20,False,Absen,2001,L,93,0,ILMU PEMERINTAHAN - S1,ILMU SOSIAL DAN ILMU POLITIK,Sarjana,2026,11


attendance_analytics.csv berhasil dibuat!
